# Chapter 5 — Classes, exceptions, modules and files

**Goal:** understand *objects*. Everything in the libraries is an object with attributes (`arr.shape`, `df.columns`)
and methods (`arr.sum()`, `clf.fit(X, y)`). You'll write a small class with the same `fit` / `predict` shape as a
scikit-learn model. You'll also handle errors, import code and read and write files.

**Where you'll meet this in MLMA Lab 1:** sections 9–11 (*classes, modules, exceptions*), and the whole Titanic part:
`clf = DecisionTreeClassifier(max_depth=3)` creates an object, and `clf.fit(...)` calls a method on it.

## 5.1 Classes and objects

A **class** is a blueprint and an **object** (instance) is one thing built from it.

- `__init__` runs when you create an object, and sets up its **attributes** (`self.something = ...`).
- **Methods** are functions defined inside the class. Their first parameter is always `self`, the object itself.
- `__repr__` controls how the object prints.

In [ ]:
class Person:
    def __init__(self, name):
        self.name = name

    def introduce(self):
        return f"Hello, my name is {self.name}."

    def __repr__(self):
        return f"Person({self.name!r})"

alice = Person("Alice")
print(alice.name)          # attribute
print(alice.introduce())   # method call: Python passes alice as `self`
print(alice)               # uses __repr__

### The pattern you'll see in every ML library

```python
model = SomeModel(hyperparameter=3)   # __init__: store settings
model.fit(X_train, y_train)            # learn from data and store results in attributes (named with a trailing _)
y_pred = model.predict(X_test)         # use the learned attributes
```

Here's a toy version: a "model" that always predicts the training mean.

In [ ]:
class MeanRegressor:
    """Predicts the mean of the training targets for every input."""

    def __init__(self):
        self.mean_ = None            # not learned yet

    def fit(self, X, y):
        self.mean_ = sum(y) / len(y)
        return self                  # returning self allows model.fit(...).predict(...)

    def predict(self, X):
        return [self.mean_ for _ in X]

model = MeanRegressor()
model.fit([[1], [2], [3]], [10, 20, 60])
print(model.mean_, model.predict([[4], [5]]))

## 5.2 Exceptions

When something goes wrong Python **raises an exception** (`ZeroDivisionError`, `KeyError`, `ValueError`, `TypeError`, …).

- Read error messages **from the bottom up**: the last line says what went wrong, the lines above say where.
- `try / except` catches an exception you expect and handles it.
- `raise ValueError("message")` signals bad input from your own code.

In [ ]:
try:
    x = 1 / 0
except ZeroDivisionError:
    print("You can't divide by zero!")

def check_probability(p):
    if not 0 <= p <= 1:
        raise ValueError(f"p must be in [0, 1], got {p}")
    return p

try:
    check_probability(1.3)
except ValueError as err:
    print("Caught:", err)

## 5.3 Modules and imports

A **module** is a file of Python code, and a **package** is a folder of modules. Four import styles, all common:

```python
import math                          # use as math.sqrt
import numpy as np                   # alias: np.array  (this alias is a universal convention)
from math import sqrt, pi            # import names directly: sqrt(2)
from numpy.linalg import inv, det    # from a submodule
```

Avoid `from module import *`: you lose track of where names come from.

Exploring an unfamiliar object: `type(obj)`, `dir(obj)` (list its attributes), `help(obj)`, or in Jupyter
`obj?` and pressing `Shift+Tab` inside the parentheses of a call.

In [ ]:
import math
from math import sqrt
import statistics as st

print(math.sqrt(2) == sqrt(2))
print(st.median([3, 1, 4, 1, 5]))
print([name for name in dir(st) if not name.startswith("_")][:8])

## 5.4 Files

Use `with open(...) as f:`. The file is closed automatically when the block ends, even if there's an error.
Modes: `"r"` read (default), `"w"` write (overwrites!), `"a"` append.
`pathlib.Path` is the modern way to build file paths.

In [ ]:
from pathlib import Path

path = Path("primer_tmp_example.txt")
with open(path, "w") as f:
    f.write("first line\n")
    f.write("second line\n")

with open(path) as f:
    for line in f:
        print(line.strip())

path.unlink()   # delete the temporary file

---
# Exercises

### Exercise 5.1: A first class
Write a class `Coin` with:
- `__init__(self, p_heads=0.5)` that stores `p_heads` as an attribute
- a method `p_tails(self)` that returns `1 - p_heads`
- a method `is_fair(self)` that returns `True` when `p_heads` is 0.5 (use `math.isclose`)

In [ ]:
import math

class Coin:
    pass  # YOUR CODE HERE

In [ ]:
c = Coin(0.7)
assert c.p_heads == 0.7 and math.isclose(c.p_tails(), 0.3) and c.is_fair() is False
assert Coin().is_fair() is True
print("Exercise 5.1 passed")

### Exercise 5.2: Methods that change state
Write a class `Counter` that tallies outcomes:
- `__init__` creates an empty dict attribute `self.counts`
- `add(self, outcome)` increases the count for `outcome` by one
- `total(self)` returns the total number of outcomes added
- `frequency(self, outcome)` returns `count / total` (or `0.0` for an outcome never seen)

In [ ]:
class Counter:
    pass  # YOUR CODE HERE

In [ ]:
tally = Counter()
for o in "HHTHT":
    tally.add(o)
assert tally.counts == {"H": 3, "T": 2} and tally.total() == 5
assert tally.frequency("H") == 0.6 and tally.frequency("X") == 0.0
print("Exercise 5.2 passed")

### Exercise 5.3: `__repr__`
Add a `__repr__` to `Interval` so that `repr(Interval(0, 1))` is exactly `"Interval(0, 1)"`.
Also add a method `contains(self, x)` that returns whether `low <= x <= high`, and a method `length(self)`.

In [ ]:
class Interval:
    def __init__(self, low, high):
        self.low = low
        self.high = high

    pass  # YOUR CODE HERE

In [ ]:
assert repr(Interval(0, 1)) == "Interval(0, 1)", f"got {repr(Interval(0, 1))}"
assert Interval(2, 5).contains(3) and not Interval(2, 5).contains(6) and Interval(2, 5).length() == 3
print("Exercise 5.3 passed")

### Exercise 5.4: A `fit` / `predict` classifier
Write `MajorityClassifier`, which learns the **most frequent label** in `y` during `fit` and predicts it for every row.
- `fit(self, X, y)` sets `self.majority_` and returns `self`
- `predict(self, X)` returns a list with one prediction per row of `X`

This is a real *baseline*: any ML model should beat it. On the Titanic data it scores about 62%.

In [ ]:
class MajorityClassifier:
    pass  # YOUR CODE HERE

In [ ]:
clf = MajorityClassifier()
assert clf.fit([[0], [1], [2], [3]], [0, 1, 0, 0]) is clf, "fit should return self"
assert clf.majority_ == 0
assert clf.predict([[5], [6]]) == [0, 0]
print("Exercise 5.4 passed")

### Exercise 5.5: Accuracy
Write a **function** (not a method) `accuracy(y_true, y_pred)` returning the fraction of positions where the two lists agree.
Then compute `baseline_acc`, the accuracy of `MajorityClassifier` trained and evaluated on the toy data below.

In [ ]:
y_train = [1, 0, 0, 1, 0, 0, 0]
y_test = [0, 1, 0, 0, 1]
X_train = [[i] for i in range(len(y_train))]
X_test = [[i] for i in range(len(y_test))]

def accuracy(y_true, y_pred):
    pass  # YOUR CODE HERE

baseline_acc = ...

In [ ]:
assert accuracy([1, 0, 1, 1], [1, 1, 1, 0]) == 0.5
assert baseline_acc == 0.6, f"got {baseline_acc}"
print("Exercise 5.5 passed")

### Exercise 5.6: Catching an exception
Write `safe_ratio(a, b)` that returns `a / b`, but returns `float("nan")` (not-a-number) when `b` is zero.
Use `try / except ZeroDivisionError`, not an `if`.

In [ ]:
def safe_ratio(a, b):
    pass  # YOUR CODE HERE

In [ ]:
assert safe_ratio(6, 3) == 2
assert math.isnan(safe_ratio(1, 0)), "expected nan"
print("Exercise 5.6 passed")

### Exercise 5.7: Raising an exception
Write `normalize(weights)` that returns a new list where the weights are divided by their sum (so they add up to 1). It should:
- raise `ValueError` if any weight is negative
- raise `ValueError` if the sum is zero

In [ ]:
def normalize(weights):
    pass  # YOUR CODE HERE

In [ ]:
assert normalize([1, 1, 2]) == [0.25, 0.25, 0.5]
for bad in ([1, -1, 2], [0, 0]):
    try:
        normalize(bad)
    except ValueError:
        pass
    else:
        raise AssertionError(f"normalize({bad}) should raise ValueError")
print("Exercise 5.7 passed")

### Exercise 5.8: Reading error messages
Each snippet in the list below fails. **Without running them**, predict which exception type each raises,
and write the names as strings in `predicted` (e.g. `"KeyError"`). The check runs them for you.

1. `{"a": 1}["b"]`
2. `[1, 2, 3][3]`
3. `int("three")`
4. `"age: " + 30`
5. `undefined_variable + 1`

Choose from: `"KeyError"`, `"IndexError"`, `"ValueError"`, `"TypeError"`, `"NameError"`.

In [ ]:
predicted = ["...", "...", "...", "...", "..."]

In [ ]:
snippets = ['{"a": 1}["b"]', '[1, 2, 3][3]', 'int("three")', '"age: " + 30', 'undefined_variable + 1']
for code, guess in zip(snippets, predicted):
    try:
        eval(code)
    except Exception as err:
        actual = type(err).__name__
        assert guess == actual, f"{code!r} raises {actual}, you predicted {guess}"
print("Exercise 5.8 passed")

### Exercise 5.9: Import styles
Using the right import statements, make these names available, then compute `result`:
- `sqrt` and `pi` imported **directly** from `math`
- the `statistics` module imported **as** `st`

`result = sqrt(pi) + st.mean([1, 2, 3])`

In [ ]:
pass  # YOUR CODE HERE
result = sqrt(pi) + st.mean([1, 2, 3])

In [ ]:
assert math.isclose(result, 3.7724538509055159)
print("Exercise 5.9 passed")

### Exercise 5.10: Exploring an object
`statistics` has a class `NormalDist`. Use `help(st.NormalDist)` or `dir(...)` to discover its methods. Then:
- create `nd`, a normal distribution with mean 100 and standard deviation 15
- `p_below_130`: P(X < 130), using its `cdf` method
- `q90`: the 90th percentile, using its `inv_cdf` method

In [ ]:
nd = ...
p_below_130 = ...
q90 = ...

In [ ]:
assert math.isclose(p_below_130, 0.9772498680518208)
assert math.isclose(q90, 119.22327348316901)
print("Exercise 5.10 passed")

### Exercise 5.11: Write and read a file
1. Write the numbers in `values` to `"primer_values.txt"`, **one per line**.
2. Read the file back into a list of **floats** called `loaded`.
3. Delete the file with `Path(...).unlink()`.

<details><summary>Hint</summary>

Write with `f.write(f"{v}\n")`. When reading, each line ends in `"\n"`, so `float(line)` still works,
but `line.strip()` is a good habit.
</details>

In [ ]:
from pathlib import Path
values = [0.25, 1.5, 3.0]
pass  # YOUR CODE HERE

In [ ]:
assert loaded == values, f"got {loaded}"
assert not Path("primer_values.txt").exists(), "remember to delete the file"
print("Exercise 5.11 passed")

### Exercise 5.12: A random-variable class
Write `DiscreteRV`, which represents a discrete random variable:
- `__init__(self, values, probs)`: store both lists, and raise `ValueError` if the probabilities don't sum to 1 (use `math.isclose`)
- `mean(self)`: $\sum x_i p_i$
- `var(self)`: $\sum (x_i - \mu)^2 p_i$
- `sample(self, rng)`: draw one value. Use `rng.choices(self.values, weights=self.probs)[0]` with a `random.Random` generator.

In [ ]:
import random

class DiscreteRV:
    pass  # YOUR CODE HERE

In [ ]:
X = DiscreteRV([0, 1, 2], [0.25, 0.5, 0.25])
assert math.isclose(X.mean(), 1) and math.isclose(X.var(), 0.5)
r = random.Random(0)
draws = [X.sample(r) for _ in range(20_000)]
assert set(draws) <= {0, 1, 2} and abs(sum(draws) / len(draws) - 1) < 0.03
try:
    DiscreteRV([0, 1], [0.5, 0.6])
except ValueError:
    pass
else:
    raise AssertionError("probabilities summing to 1.1 should raise ValueError")
print("Exercise 5.12 passed")

---
**Chapter 5 done.** That's all the core Python you need. From now on it's libraries, starting with NumPy in Chapter 6.